In this notebook we process the data from the MMPBSA experiments by removing any duplicates

In [22]:
import pandas as pd

# We drop all duplicate molecules and keep the best (=lowest) score
def clean_dataset(df):
    n_duplicates = df["smiles"].duplicated().sum()
    print(f"Dataset contains {n_duplicates} duplicates.")
    
    assert not df["target"].isna().any(), "Dataset contains molecules without a score."
    
    clean_df = df.sort_values(by="target").drop_duplicates(["smiles"], keep="first")
    # Avoid any information leakage by having the data ordered
    clean_df = clean_df.sample(frac=1.0, random_state=0).reset_index(drop=True)
    return clean_df

We start by cleaning the Enamine datasets that have been used in the literature to evaluate active learning.

In [ ]:
for ds in ["unprocessed_Enamine10k_scores.csv", "unprocessed_Enamine50k_scores.csv"]:
    df = pd.read_csv(ds)
    clean_df = clean_dataset(df)
    
    clean_df.to_csv(ds.removeprefix("unprocessed_"), index=False)

Dataset contains 3 duplicates.
Dataset contains 7 duplicates.


First we need to extract the MMPBSA results. We also average the score over all runs for each molecule. This will produce `benchmark.csv`, `benchmark_dg_en_gb_avg.csv`, `mmpbsa.csv` and `mmpbsa_dg_en_gb_avg.csv`.
`benchmark.csv` and `mmpbsa.csv` contain the full information from the simulations. From now on we will work with the averaged values.

In [16]:
!./unpack_and_combine_mmpbsa.sh

In [ ]:
mmpbsa = pd.read_csv("mmpbsa_dg_en_gb_avg.csv")
mmpbsa_clean = clean_dataset(mmpbsa)

mmpbsa_clean[["smiles", "target", "name"]].to_csv("MCL1-mmpbsa-gb_scores.csv", index=False)

Dataset contains 554 duplicates.


We also clean the docking scores

In [13]:
!cp ../docking/results.csv vina.csv

In [41]:
docking = pd.read_csv("vina.csv", names=["name", "smiles", "target"])
docking_clean = clean_dataset(docking)

docking_clean[["smiles", "target", "name"]].to_csv("MCL1-vina_scores.csv", index=False)

Dataset contains 555 duplicates.
